# 02 基线模型

## 本文件目标
在初始数据底座上建立第一个可运行的时间序列 baseline。

## 本文件主要工作
1. 从 date 中提取基础时间特征
2. 使用最近 365 天数据建模
3. 使用最后 16 天作为验证集
4. 使用 CatBoost 训练 baseline
5. 生成 baseline submission

## 为什么这样做
- 使用最近 365 天是为了降低训练成本，并优先利用较新的数据
- 使用最后 16 天验证，是为了模拟真实未来预测场景
- 使用 CatBoost，是因为类别特征较多，处理更方便

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(PATH + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(PATH + "transactions.csv", parse_dates=["date"])
sample_submission = pd.read_csv(PATH + "sample_submission.csv")

In [ ]:
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

In [ ]:
train_df = base_train.copy()
test_df = base_test.copy()

for df in [train_df, test_df]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
num_cols = ["onpromotion", "dcoilwtico", "year", "month", "day", "dayofweek", "dayofyear", "is_weekend"]
feature_cols = cat_cols + num_cols
target_col = "sales"

for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

In [ ]:
cutoff_date = train_df["date"].max() - pd.Timedelta(days=365)
train_df_small = train_df[train_df["date"] >= cutoff_date].copy()

last_date = train_df_small["date"].max()
val_start_date = last_date - pd.Timedelta(days=15)

print("train_df_small shape:", train_df_small.shape)
print("train_df_small date range:", train_df_small["date"].min(), "to", train_df_small["date"].max())
print("train max date:", last_date)
print("validation start date:", val_start_date)

train_part = train_df_small[train_df_small["date"] < val_start_date].copy()
valid_part = train_df_small[train_df_small["date"] >= val_start_date].copy()

print("train_part shape:", train_part.shape)
print("valid_part shape:", valid_part.shape)
print("train_part date range:", train_part["date"].min(), "to", train_part["date"].max())
print("valid_part date range:", valid_part["date"].min(), "to", valid_part["date"].max())

In [ ]:
X_train = train_part[feature_cols].copy()
X_valid = valid_part[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = np.log1p(train_part[target_col].values)
y_valid = np.log1p(valid_part[target_col].values)

model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=50
)

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

valid_pred_log = model.predict(X_valid)
valid_pred = np.expm1(valid_pred_log)
valid_pred = np.clip(valid_pred, 0, None)

valid_true = valid_part[target_col].values
score = rmsle(valid_true, valid_pred)
print("Validation RMSLE:", score)

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "id": test_df["id"],
    "sales": test_pred
})

submission.to_csv("submission_baseline.csv", index=False)
print(submission.head())
print("submission_baseline.csv saved!")

## 当前结果
- experiment_name: `baseline_catboost_recent365`
- features_used: `stores + oil + basic_date_features`
- validation_split: `last_16_days`
- model: `CatBoostRegressor`
- params: `iterations=300, depth=6, lr=0.05`
- score: `0.686549`

## 本阶段小结
基础日期特征 + stores + oil 的 baseline 可以稳定跑通，但分数还有明显提升空间。